# Demo Notebook

## Simple request

In [ ]:
import requests

In [ ]:
url = "http://localhost:5001/rate?ccy_pair=GBPUSD"
response = requests.get(url)
print(response.text)

In [ ]:
url = "http://localhost:5001/rate"
response = requests.get(url)
print(response.text)

In [ ]:
url = "http://localhost:5001/rate"
response = requests.get(url)
print(response.text)

In [ ]:
url = "http://localhost:5001/rate?ccy_pair=GBPJPY"
response = requests.get(url)
print(response.text)

In [ ]:
url = "http://localhost:8000/convert"
payload = {"ccy_from": "GBP", "ccy_to": "USD", "quantity": 100}
response = requests.post(url, json=payload)
print(f"Status code: {response.status_code}")
print(f"Response text: {response.text}")
if response.ok:
    print(response.json())

In [ ]:
payload = {"ccy_from": "EUR", "ccy_to": "JPY", "quantity": 100}
response = requests.post(url, json=payload)
print(response.json())

In [ ]:
payload = {"ccy_from": "JPY", "ccy_to": "CAD", "quantity": 100000}
response = requests.post(url, json=payload)
print(response.json())

In [ ]:
payload = {"ccy_from": "USD", "ccy_to": "CHF", "quantity": 25000}
response = requests.post(url, json=payload)
print(response.json())

In [ ]:
payload = {"ccy_from": "AUD", "ccy_to": "NZD", "quantity": 100000}
response = requests.post(url, json=payload)
print(response.json())

## Query with functions

In [ ]:
import requests

def get_fx_rate(ccy_pair: str) -> float:
    response = requests.get(f"http://localhost:5001/rate?ccy_pair={ccy_pair}")
    response.raise_for_status()
    return float(response.text)

# Examples
eur_usd = get_fx_rate("EURUSD")
print(f"EUR/USD: {eur_usd}")  # EUR/USD: 1.10

gbp_usd = get_fx_rate("GBPUSD")
print(f"GBP/USD: {gbp_usd}")  # GBP/USD: 1.28

In [ ]:
import httpx
import asyncio
from typing import Dict, List, Tuple

class CurrencyClient:
    def __init__(self, base_url: str = "http://localhost:8000"):
        self.base_url = base_url
        self.client = httpx.AsyncClient()

    async def convert_currency(self, from_ccy: str, to_ccy: str, quantity: float) -> Dict:
        """
        Convert currency using the async API.
        """
        response = await self.client.post(
            f"{self.base_url}/convert",
            json={
                "ccy_from": from_ccy,
                "ccy_to": to_ccy,
                "quantity": quantity
            }
        )
        response.raise_for_status()
        return response.json()

    async def get_fx_rate(self, ccy_pair: str) -> float:
        """
        Get FX rate from the mock service.
        """
        response = await self.client.get(
            "http://localhost:5001/rate",
            params={"ccy_pair": ccy_pair}
        )
        response.raise_for_status()
        return float(response.text)

    async def batch_convert(self, conversions: List[Tuple[str, str, float]]) -> List[Dict]:
        """
        Perform multiple currency conversions concurrently.
        """
        tasks = [
            self.convert_currency(from_ccy, to_ccy, amount)
            for from_ccy, to_ccy, amount in conversions
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    async def close(self):
        await self.client.aclose()

# Usage example
async def main():
    client = CurrencyClient()
    try:
        # Single conversion
        result = await client.convert_currency("EUR", "USD", 100)
        print(f"EUR to USD: {result}")

        # Multiple concurrent conversions
        conversions = [
            ("EUR", "USD", 100),
            ("GBP", "JPY", 50),
            ("USD", "EUR", 75)
        ]
        results = await client.batch_convert(conversions)
        for conv, result in zip(conversions, results):
            if isinstance(result, Exception):
                print(f"Error converting {conv}: {result}")
            else:
                print(f"{conv[0]} to {conv[1]}: {result}")
    finally:
        await client.close()

# Run with:
# asyncio.run(main())   # In python script
# await main()          # In Jupyter notebook

In [ ]:
await main()

# More examples

## Synchronous query

In [ ]:
import requests

def convert_currency_sync(from_ccy: str, to_ccy: str, amount: float):
    response = requests.post(
        "http://localhost:8000/convert",
        json={
            "ccy_from": from_ccy,
            "ccy_to": to_ccy,
            "quantity": amount
        }
    )
    response.raise_for_status()
    return response.json()

# Usage examples with traditional method:
# EUR to USD
result = convert_currency_sync("EUR", "USD", 100)
print(result)  # {"quantity": 110.0, "ccy": "USD"}

# GBP to JPY (using triangulation)
result = convert_currency_sync("GBP", "JPY", 100)
print(result)  # {"quantity": 18560.0, "ccy": "JPY"}

# With error handling
try:
    result = convert_currency_sync("EUR", "USD", 100)
    print(f"Converted amount: {result['quantity']} {result['ccy']}")
except requests.exceptions.RequestException as e:
    print(f"Error during conversion: {e}")

## Asynchronous query

In [ ]:
import httpx

async def convert_currency(from_ccy: str, to_ccy: str, amount: float):
    async with httpx.AsyncClient() as client:
        response = await client.post(
            "http://localhost:8000/convert",
            json={
                "ccy_from": from_ccy,
                "ccy_to": to_ccy,
                "quantity": amount
            }
        )
        return response.json()

In [ ]:
# Usage examples:
# EUR to USD conversion
result_1 = await convert_currency("EUR", "USD", 100)
# GBP to JPY conversion (using triangulation)
result_2 = await convert_currency("GBP", "JPY", 100)

print(result_1)  # {"quantity": 110.0, "ccy": "USD"}
print(result_2)  # {"quantity": 18560.0, "ccy": "JPY"}

## Error Handling

### Traditional Error Handling (Synchronous)

In [ ]:
import requests
from requests.exceptions import RequestException
from typing import Dict, Any

def safe_convert_currency_sync(from_ccy: str, to_ccy: str, amount: float) -> Dict[str, Any]:
    try:
        response = requests.post(
            "http://localhost:8000/convert",
            json={
                "ccy_from": from_ccy,
                "ccy_to": to_ccy,
                "quantity": amount
            },
            timeout=5  # Add timeout for safety
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 400:
            print(f"Validation error: {e.response.json()['detail']}")
        elif e.response.status_code == 422:
            print("Invalid input format")
        elif e.response.status_code == 500:
            print("Service temporarily unavailable")
        raise
    except requests.exceptions.Timeout:
        print("Request timed out")
        raise
    except requests.exceptions.ConnectionError:
        print("Connection error - service may be down")
        raise
    except RequestException as e:
        print(f"An error occurred: {e}")
        raise

# Usage example with error handling
try:
    # Valid conversion
    result = safe_convert_currency_sync("EUR", "USD", 100)
    print(f"Converted amount: {result['quantity']} {result['ccy']}")

    # Invalid currency
    result = safe_convert_currency_sync("XXX", "USD", 100)
except RequestException as e:
    print(f"Request failed: {e}")

# Batch processing with error handling
def batch_convert_sync(conversions: list[tuple[str, str, float]]) -> list[Dict[str, Any]]:
    results = []
    for from_ccy, to_ccy, amount in conversions:
        try:
            result = safe_convert_currency_sync(from_ccy, to_ccy, amount)
            results.append(result)
        except RequestException as e:
            print(f"Failed to convert {amount} {from_ccy} to {to_ccy}: {e}")
    return results

# Batch conversion example
conversions = [
    ("EUR", "USD", 100),
    ("GBP", "JPY", 50),
    ("USD", "EUR", 75)
]
results = batch_convert_sync(conversions)

## Additional Traditional Examples

### Using Sessions for Multiple Requests

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Create a session with retry strategy
def create_currency_session(retries=3):
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=0.5,  # Wait 0.5, 1, 2... seconds between retries
        status_forcelist=[500, 502, 503, 504]  # Retry on these HTTP status codes
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

# Use session for better performance with multiple requests
def convert_with_session(conversions: list[tuple[str, str, float]]):
    results = []
    with create_currency_session() as session:
        for from_ccy, to_ccy, amount in conversions:
            try:
                response = session.post(
                    "http://localhost:8000/convert",
                    json={
                        "ccy_from": from_ccy,
                        "ccy_to": to_ccy,
                        "quantity": amount
                    },
                    timeout=5
                )
                response.raise_for_status()
                results.append(response.json())
            except requests.exceptions.RequestException as e:
                print(f"Error converting {from_ccy} to {to_ccy}: {e}")
    return results

# Example usage with session
pairs = [
    ("EUR", "USD", 100),
    ("EUR", "USD", 200),
    ("EUR", "USD", 300)
]
results = convert_with_session(pairs)

### Parallel Processing for Large Batches

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple

def convert_currency_parallel(conversions: List[Tuple[str, str, float]], 
                            max_workers: int = 5) -> List[Dict[str, Any]]:
    """
    Convert multiple currency pairs in parallel using threads.
    
    Args:
        conversions: List of (from_currency, to_currency, amount) tuples
        max_workers: Maximum number of parallel threads
        
    Returns:
        List of conversion results
    """
    results = []
    session = create_currency_session()

    def convert_single(conv: Tuple[str, str, float]) -> Dict[str, Any]:
        from_ccy, to_ccy, amount = conv
        response = session.post(
            "http://localhost:8000/convert",
            json={
                "ccy_from": from_ccy,
                "ccy_to": to_ccy,
                "quantity": amount
            }
        )
        response.raise_for_status()
        return response.json()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_conv = {
            executor.submit(convert_single, conv): conv 
            for conv in conversions
        }
        
        for future in as_completed(future_to_conv):
            conv = future_to_conv[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                print(f"Conversion failed for {conv}: {e}")
    
    return results

# Example usage with parallel processing
large_batch = [
    ("EUR", "USD", amount) 
    for amount in range(100, 1100, 100)
]
parallel_results = convert_currency_parallel(large_batch)

### CSV Processing Example

Users should be able to send GET requests to the Currency Converter REST API with parameters ccy_from=USD&ccy_to=GBP&quantity=1000 and receive a response in the JSON format {“quantity”: 779.77, “ccy”: “GBP”}.

The service should be using near-real-time FX rates that can be derived from the CoinDesk REST API (https://api.coindesk.com/v1/bpi/currentprice.json). This service provides real-time bitcoin prices in 3 currencies (USD, EUR, GBP)

In [ ]:
import csv
import pandas as pd
from datetime import datetime

def process_currency_file(input_file: str, output_file: str):
    """
    Process a CSV file containing currency conversion requests.
    
    Expected CSV format:
    from_currency,to_currency,amount
    EUR,USD,100
    GBP,JPY,200
    ...
    """
    # Read conversions from CSV
    df = pd.read_csv(input_file)
    
    # Perform conversions
    results = []
    session = create_currency_session()
    
    for _, row in df.iterrows():
        try:
            response = session.post(
                "http://localhost:8000/convert",
                json={
                    "ccy_from": row["from_currency"],
                    "ccy_to": row["to_currency"],
                    "quantity": float(row["amount"])
                }
            )
            response.raise_for_status()
            result = response.json()
            results.append({
                "from_currency": row["from_currency"],
                "to_currency": row["to_currency"],
                "original_amount": row["amount"],
                "converted_amount": result["quantity"],
                "conversion_time": datetime.now().isoformat()
            })
        except Exception as e:
            print(f"Error processing row {row}: {e}")
    
    # Save results
    pd.DataFrame(results).to_csv(output_file, index=False)
    return results

# Example usage
file_results = process_currency_file(
    "conversions.csv",
    f"conversion_results_{datetime.now():%Y%m%d_%H%M%S}.csv"
)

### Rate Monitoring Example

In [ ]:
import time
from typing import Dict, List

def monitor_exchange_rates(currency_pairs: List[Tuple[str, str]], 
                         interval: int = 60,
                         duration: int = 3600):
    """
    Monitor exchange rates for specified currency pairs.
    
    Args:
        currency_pairs: List of (from_currency, to_currency) tuples
        interval: Seconds between checks
        duration: Total monitoring duration in seconds
    """
    session = create_currency_session()
    start_time = time.time()
    rate_history: Dict[str, List[Dict]] = {
        f"{from_ccy}/{to_ccy}": [] 
        for from_ccy, to_ccy in currency_pairs
    }
    
    while time.time() - start_time < duration:
        for from_ccy, to_ccy in currency_pairs:
            try:
                response = session.post(
                    "http://localhost:8000/convert",
                    json={
                        "ccy_from": from_ccy,
                        "ccy_to": to_ccy,
                        "quantity": 1
                    }
                )
                response.raise_for_status()
                result = response.json()
                rate_history[f"{from_ccy}/{to_ccy}"].append({
                    "time": datetime.now().isoformat(),
                    "rate": result["quantity"]
                })
                print(f"{from_ccy}/{to_ccy}: {result['quantity']}")
            except Exception as e:
                print(f"Error monitoring {from_ccy}/{to_ccy}: {e}")
        
        time.sleep(interval)
    
    return rate_history

# Example usage
pairs_to_monitor = [
    ("EUR", "USD"),
    ("GBP", "USD"),
    ("USD", "JPY")
]
rate_history = monitor_exchange_rates(
    pairs_to_monitor,
    interval=60,    # Check every minute
    # duration=3600   # Monitor for 1 hour
    duration=120   # Monitor for 2 minutes
)

### Modern Service Errors (Asynchronous)

In [ ]:
import httpx
from typing import Optional, Dict, Any

class FXServiceError(Exception):
    """Base exception for FX service errors"""
    pass

class RateNotFoundError(FXServiceError):
    """Raised when a rate is not available"""
    pass

async def get_fx_rate_safe(ccy_pair: str) -> Optional[float]:
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(
                f"http://localhost:5001/rate",
                params={"ccy_pair": ccy_pair}
            )
            response.raise_for_status()
            return float(response.text)
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 404:
            raise RateNotFoundError(f"Rate not found for {ccy_pair}")
        raise FXServiceError(f"FX service error: {e}")
    except httpx.RequestError as e:
        raise FXServiceError(f"Connection error: {e}")
    except ValueError as e:
        raise FXServiceError(f"Invalid rate format: {e}")

#### Conversion API Errors
async def safe_convert_currency(from_ccy: str, to_ccy: str, amount: float) -> Dict[str, Any]:
    async with httpx.AsyncClient() as client:
        try:
            response = await client.post(
                "http://localhost:8000/convert",
                json={
                    "ccy_from": from_ccy,
                    "ccy_to": to_ccy,
                    "quantity": amount
                }
            )
            response.raise_for_status()
            return response.json()
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 400:
                print(f"Validation error: {e.response.json()['detail']}")
            elif e.response.status_code == 422:
                print("Invalid input format")
            elif e.response.status_code == 500:
                print("Service temporarily unavailable")
            raise

In [ ]:
# await safe_convert_currency("EUR", "USD1", 100)
# await safe_convert_currency("EUR", "US1", 100)
# await safe_convert_currency("EUR", "USD", 100)